# Cross-Modal Diagnostic Observability — Stage 11C-R2

## tl;dr

This notebook preserves the sealed Stage 11C-R result and performs a new, outcome-free readjudication of the same five manual official receipts. It corrects two identified evidence gaps in v0.1: outer-ZIP-only inspection and filename-only label detection.

## Context & Methods

Stage 11C-R v0.1 received all five official payloads but qualified only one domain. Review showed that v0.1 did not recurse into nested archives, did not read released CSV/XLSX/README content, and did not apply publisher-documented mask semantics or author-documented patient prefixes.

### Key assumptions and boundaries

- Only the five already-received original development candidates are inspected.
- The manually frozen provider-evidence registry below was reviewed before any model performance was computed.
- No reserve dataset, embedding, source AUC, transfer result, DDO2 operation, Stage 12 operation, or locked-blind asset is accessed.
- The previous Stage 11C-R record remains immutable; this notebook writes a separate Stage 11C-R2 record.
- A dataset qualifies only if its pixel payload, per-sample diagnostic labels, and deterministic patient/lesion groups are all auditable.


In [1]:
# @title 11C-R2-0. Mount Drive, verify sealed lineage, and freeze provider evidence
import hashlib, io, json, os, re, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd

try:
    from PIL import Image
except Exception:
    Image = None
try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

DEFAULT_ROOT = Path('/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability') if IN_COLAB else Path('/tmp/Cross-Modal_Diagnostic_Observability')
PROJECT_ROOT = Path(os.environ.get('CDO_PROJECT_ROOT', str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT/'05_Code'/'Cross_Modal'
CM_ROOT = PROJECT_ROOT/'06_Data_Records'/'Cross_Modal'
R1_ROOT = CM_ROOT/'Stage11C-R_Manual_Official_Receipt_Validation_And_Typed_Hold_Readjudication_v0.1'
ROOT = CM_ROOT/'Stage11C-R2_Recursive_Payload_And_Provider_Evidence_Readjudication_v0.1'
RECEIPT_ROOT = PROJECT_ROOT/'00_Data_Acquisition'/'Stage11C_Manual_Official_Receipts'
P0,P1,P2,P3,P4,P5 = [ROOT/x for x in ['00_Protocol','01_Recursive_Inventory','02_Evidence_Adapters','03_Readjudication','04_Firewall','05_Results']]
for p in [CODE_ROOT,P0,P1,P2,P3,P4,P5]: p.mkdir(parents=True,exist_ok=True)

NOTEBOOK_NAME='CrossModal_Stage11C-R2_Recursive_Payload_And_Provider_Evidence_Readjudication_v0.1.ipynb'
NOTEBOOK_PATH=CODE_ROOT/NOTEBOOK_NAME
R1_FINAL=R1_ROOT/'05_Results'/'Stage11C-R_Complete_v0.1.json'
R1_LEDGER=R1_ROOT/'01_Receipt_Integrity'/'Stage11C-R_Manual_Official_Receipt_Ledger_v0.1.csv'
R1_DECISIONS=R1_ROOT/'03_Readjudication'/'Stage11C-R_Typed_Hold_Readjudication_v0.1.csv'
EXPECTED_R1_FINAL='d50440479a53b63cc22c8bd643089ce43545587f29b0062897f76b9e774ac768'
DATASETS=['BUS_BRA_2024','BUSI_WHU_2025_V3','BREAST_LESIONS_USG_2024','BUS_UCLM_2025_V3','RODRIGUES_BUI_2017']
FOLDERS={'BUS_BRA_2024':'BUS_BRA','BUSI_WHU_2025_V3':'BUSI_WHU','BREAST_LESIONS_USG_2024':'BREAST_LESIONS_USG','BUS_UCLM_2025_V3':'BUS_UCLM','RODRIGUES_BUI_2017':'RODRIGUES_BUI'}
LOCKED=['BUSI_CAIRO_2019','OASBUD_2017','DERM7PT_2019']
MINIMUM=4

PROTOCOL=P0/'Stage11C-R2_Protocol_Seal_v0.1.json'
PROVIDER_REGISTRY=P0/'Stage11C-R2_Frozen_Provider_Evidence_Registry_v0.1.json'
PARENT_COMMIT=P0/'Stage11C-R2_Parent_Input_Commitment_v0.1.csv'
RECEIPT_REVERIFY=P1/'Stage11C-R2_Receipt_Reverification_v0.1.csv'
INVENTORY=P1/'Stage11C-R2_Recursive_Archive_Inventory_v0.1.csv'
ARCHIVE_SUMMARY=P1/'Stage11C-R2_Recursive_Archive_Summary_v0.1.csv'
TABLE_AUDIT=P2/'Stage11C-R2_Released_Table_Audit_v0.1.csv'
EVIDENCE_MATRIX=P2/'Stage11C-R2_Dataset_Evidence_Matrix_v0.1.csv'
DECISIONS=P3/'Stage11C-R2_Typed_Readjudication_v0.1.csv'
ROSTER=P3/'Stage11C-R2_Qualified_Development_Roster_v0.1.csv'
HANDOFF=P3/'Stage11C-R2_Stage11D-R_Handoff_v0.1.json'
FIREWALL=P4/'Stage11C-R2_Independent_Validity_And_Firewall_Checks_v0.1.csv'
REPORT=P5/'Stage11C-R2_Provider_Evidence_And_Recursive_Payload_Report_v0.1.md'
MANIFEST=P5/'Stage11C-R2_Output_Integrity_Manifest_v0.1.csv'
FINAL=P5/'Stage11C-R2_Complete_v0.1.json'
RUNTIME=P5/'Stage11C-R2_Runtime_State_v0.1.json'

def now(): return datetime.now(timezone.utc).isoformat()
def sha_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def sha_bytes(b): return hashlib.sha256(b).hexdigest()
def sha_json(x): return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def canon(df): return df.fillna('').to_csv(index=False,lineterminator='\n',float_format='%.12g')
def markdown_table(df):
    x=df.fillna('').astype(str)
    esc=lambda s:str(s).replace('|','\\|').replace('\n',' ')
    header='| '+' | '.join(esc(c) for c in x.columns)+' |'
    rule='| '+' | '.join('---' for _ in x.columns)+' |'
    body=['| '+' | '.join(esc(v) for v in row)+' |' for row in x.itertuples(index=False,name=None)]
    return '\n'.join([header,rule]+body)
def write_text(p,s):
    p=Path(p)
    if p.exists(): assert p.read_text(encoding='utf-8')==s, f'Replay mismatch: {p}'
    else: p.write_text(s,encoding='utf-8')
def write_csv(p,d): write_text(p,canon(d))
def write_json(p,x): write_text(p,json.dumps(x,indent=2,ensure_ascii=False)+'\n')
def verify_self(p,field,expected=None):
    x=json.loads(Path(p).read_text(encoding='utf-8')); claimed=x[field]; y=dict(x); y.pop(field)
    assert sha_json(y)==claimed, f'Self-hash mismatch: {p}'
    if expected: assert claimed==expected, f'Unexpected parent hash: {p}'
    return x

required=[NOTEBOOK_PATH,R1_FINAL,R1_LEDGER,R1_DECISIONS]
missing=[str(p) for p in required if not p.is_file()]
assert not missing,'Missing sealed inputs:\n'+'\n'.join(missing)
r1=verify_self(R1_FINAL,'final_record_sha256',EXPECTED_R1_FINAL)
assert r1['decision']=='HOLD_STAGE11C_R_INSUFFICIENT_QUALIFIED_MANUAL_RECEIPTS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
assert r1['stage11d_r_authorised'] is False and r1['locked_blind_assets_touched'] is False
assert sha_file(R1_LEDGER)==r1['manual_receipt_ledger_sha256'],'Parent receipt ledger hash mismatch'
assert sha_file(R1_DECISIONS)==r1['typed_readjudication_sha256'],'Parent readjudication hash mismatch'
r1_ledger=pd.read_csv(R1_LEDGER); r1_decisions=pd.read_csv(R1_DECISIONS)
assert r1_decisions.dataset_id.tolist()==DATASETS
REPLAY=FINAL.is_file()

provider_evidence={
 'reviewed_utc':'2026-07-22T00:00:00+00:00',
 'review_boundary':'MANUAL_OFFICIAL_PROVIDER_AND_AUTHOR_EVIDENCE_ONLY_NO_MODEL_OUTCOME',
 'datasets':{
  'BUS_BRA_2024':{
   'receipt_version':'Zenodo 1.0','doi':'10.5281/zenodo.8231412','provider_url':'https://zenodo.org/records/8231412','author_code_url':'https://github.com/wgomezf/BUS-BRA','license':'CC BY 4.0','expected_original_images':1875,'expected_groups':1064,'expected_classes':['benign','malignant'],'label_semantics':'biopsy-proven benign/malignant plus BI-RADS 2-5; released CSV files carry dataset information','grouping_semantics':'1064 anonymized patients; internal table must expose a reproducible patient/case key'},
  'BUSI_WHU_2025_V3':{
   'receipt_version':'Mendeley version 3','doi':'10.17632/k6cpmwybk3.3','provider_url':'https://data.mendeley.com/datasets/k6cpmwybk3/3','license':'CC BY 4.0','expected_original_images':927,'expected_groups':None,'expected_classes':['benign','malignant'],'label_semantics':'publisher states 927 benign/malignant breast ultrasound images; internal payload must map each image to a class','grouping_semantics':'lesion/image grouping is accepted only when released payload deterministically pairs each original image and its annotation or exposes a case key'},
  'BREAST_LESIONS_USG_2024':{
   'receipt_version':'TCIA release dated 2023-12-15','doi':'10.7937/7BVR-N342','provider_url':'https://www.cancerimagingarchive.net/collection/breast-lesions-usg/','license':'TCIA Data Usage Policy / collection license','expected_original_images':256,'expected_groups':256,'expected_classes':['benign','malignant'],'label_semantics':'clinical workbook exposes BIRADS and Diagnosis','grouping_semantics':'256 scans from 256 patients; clinical workbook identifier is mandatory'},
  'BUS_UCLM_2025_V3':{
   'receipt_version':'manual receipt obtained through frozen Mendeley v1 route; payload reconciled against official counts','doi':'10.17632/7fvgj4jsp7.1','provider_url':'https://data.mendeley.com/datasets/7fvgj4jsp7/1','current_metadata_url':'https://data.mendeley.com/datasets/7fvgj4jsp7/3','author_code_url':'https://github.com/noeliavallez/BUS-UCLM-Dataset','license':'CC BY-NC 3.0 on v1 receipt page','expected_original_images':683,'expected_groups':38,'expected_class_counts':{'benign':174,'malignant':90,'normal':419},'expected_classes':['benign','malignant','normal'],'label_semantics':'publisher documents RGB masks: green benign, red malignant, black normal','grouping_semantics':'author prepare_partitions.py defines patient/case as first four filename characters'},
  'RODRIGUES_BUI_2017':{
   'receipt_version':'Mendeley version 1','doi':'10.17632/wmy84gzngw.1','provider_url':'https://data.mendeley.com/datasets/wmy84gzngw/1','license':'CC BY 4.0','expected_original_images':250,'expected_groups':None,'expected_class_counts':{'benign':100,'malignant':150},'expected_classes':['benign','malignant'],'label_semantics':'publisher states 100 benign and 150 malignant images; internal payload must map each image to a class','grouping_semantics':'no patient count is asserted by provider; qualify only if payload exposes a deterministic case/lesion key'}
 }}
provider_evidence['registry_sha256']=sha_json(provider_evidence)
if REPLAY:
    old=verify_self(PROVIDER_REGISTRY,'registry_sha256')
    assert old==provider_evidence
else: write_json(PROVIDER_REGISTRY,provider_evidence)

commit=pd.DataFrame([{'role':p.name,'relative_path':str(p.relative_to(PROJECT_ROOT)),'size_bytes':p.stat().st_size,'sha256':sha_file(p)} for p in required[1:]])
if REPLAY: assert canon(pd.read_csv(PARENT_COMMIT))==canon(commit)
else: write_csv(PARENT_COMMIT,commit)
spec={'scope':'SAME_FIVE_ORIGINAL_MANUAL_OFFICIAL_RECEIPTS','new_evidence':['recursive nested-archive inventory','released CSV/XLSX/README content','frozen official provider descriptions','publisher mask semantics','author-documented patient prefix'], 'qualification_gates':['exact receipt bytes','safe readable payload','auditable per-sample diagnostic label','deterministic patient or lesion group'], 'prohibited':['reserve datasets','performance-based substitution','embedding','source AUC','transfer result','DDO2','Stage12','locked-blind assets']}
payload={'stage':'Stage11C-R2','version':'0.1','parent_stage11c_r_final_sha256':r1['final_record_sha256'],'parent_receipt_ledger_sha256':sha_file(R1_LEDGER),'parent_typed_readjudication_sha256':sha_file(R1_DECISIONS),'provider_registry_sha256':provider_evidence['registry_sha256'],'analysis_spec':spec}
if REPLAY:
    seal=verify_self(PROTOCOL,'seal_sha256')
    for k,v in payload.items(): assert seal[k]==v
else:
    seal=dict(payload); seal['sealed_utc']=now(); seal['seal_sha256']=sha_json(seal); write_json(PROTOCOL,seal)
runtime={'stage':'Stage11C-R2','replay_mode':REPLAY,'manual_receipts_only':True,'provider_requests_during_runtime':False,'manual_provider_evidence_review':True,'reserves_accessed':False,'embeddings_computed':False,'performance_evaluated':False,'ddo2_fitted':False,'stage12_authorised':False,'locked_blind_assets_touched':False}
print('Parent Stage11C-R verified:',r1['final_record_sha256']); print('Provider registry / Protocol seal / Replay:',provider_evidence['registry_sha256'],seal['seal_sha256'],REPLAY)


Mounted at /content/drive
Parent Stage11C-R verified: d50440479a53b63cc22c8bd643089ce43545587f29b0062897f76b9e774ac768
Provider registry / Protocol seal / Replay: 248b85844bbcb2c8dcad75e6326aa483ce9224047fee934fcd7da2602bacdc3b 12ca6c10e436e75efbb5bbca9e956f14f877bc1b8c646867060c9f8a6c326e25 False


In [2]:
# @title 11C-R2-1. Reverify exact receipts and recursively inventory nested payloads
IMAGE_EXT={'.png','.jpg','.jpeg','.bmp','.tif','.tiff','.dcm'}
TABLE_EXT={'.csv','.tsv','.xlsx','.xls','.json','.txt','.xml','.md'}
ARCHIVE_EXT={'.zip','.rar','.7z','.tar','.gz'}
MAX_NESTED_BYTES=512*1024*1024

expected={(r.dataset_id,r.file_name):(int(r.size_bytes),str(r.sha256)) for r in r1_ledger.itertuples()}
reverify=[]; outer_paths={}; inventory_rows=[]; opaque_rows=[]
for d in DATASETS:
    folder=RECEIPT_ROOT/FOLDERS[d]
    files=sorted([p for p in folder.iterdir() if p.is_file() and not p.name.startswith('.')]) if folder.is_dir() else []
    outer_paths[d]=files
    for p in files:
        key=(d,p.name); observed_sha=sha_file(p); observed_size=p.stat().st_size
        reverify.append({'dataset_id':d,'file_name':p.name,'size_bytes':observed_size,'sha256':observed_sha,'present_in_r1_ledger':key in expected,'size_matches_r1':key in expected and observed_size==expected[key][0],'sha256_matches_r1':key in expected and observed_sha==expected[key][1]})
reverify=pd.DataFrame(reverify)
assert len(reverify)==len(expected) and reverify[['present_in_r1_ledger','size_matches_r1','sha256_matches_r1']].all().all(),'Receipt bytes changed since Stage11C-R'
if REPLAY: assert canon(pd.read_csv(RECEIPT_REVERIFY))==canon(reverify)
else: write_csv(RECEIPT_REVERIFY,reverify)

entry_lookup={}
def safe_name(name):
    q=PurePosixPath(str(name).replace('\\','/')); parts=q.parts
    return not q.is_absolute() and '..' not in parts and not (parts and re.match(r'^[A-Za-z]:',parts[0]))
def read_chain(outer,chain):
    data=None
    for member in chain:
        source=outer if data is None else io.BytesIO(data)
        with zipfile.ZipFile(source) as z: data=z.read(member)
    return data
def scan_zip(d,outer,source,chain_prefix=(),depth=0):
    with zipfile.ZipFile(source) as z:
        for info in z.infolist():
            if info.is_dir(): continue
            chain=tuple(chain_prefix)+(info.filename,); virtual='!/'.join(chain); suffix=Path(info.filename).suffix.lower()
            header=b''
            try:
                with z.open(info) as f: header=f.read(8)
            except Exception: pass
            is_nested_zip=(suffix=='.zip' or header.startswith(b'PK\x03\x04')) and info.file_size<=MAX_NESTED_BYTES
            is_rar=suffix=='.rar' or header.startswith(b'Rar!')
            row={'dataset_id':d,'outer_archive':outer.name,'depth':depth,'virtual_path':virtual,'member_name':info.filename,'suffix':suffix,'size_bytes':int(info.file_size),'compressed_bytes':int(info.compress_size),'path_safe':safe_name(info.filename),'encrypted':bool(info.flag_bits & 1),'nested_zip':is_nested_zip,'opaque_rar_or_7z':bool(is_rar or suffix=='.7z'),'sha256_if_small':''}
            inventory_rows.append(row); entry_lookup[(d,virtual)]={'outer':outer,'chain':chain,'suffix':suffix,'size':int(info.file_size)}
            if is_nested_zip:
                try:
                    data=z.read(info); scan_zip(d,outer,io.BytesIO(data),chain,depth+1)
                except Exception as e:
                    opaque_rows.append({'dataset_id':d,'virtual_path':virtual,'reason':'NESTED_ZIP_READ_FAILED '+type(e).__name__+': '+str(e)[:200]})
            elif is_rar or suffix=='.7z':
                opaque_rows.append({'dataset_id':d,'virtual_path':virtual,'reason':'OPAQUE_NON_ZIP_ARCHIVE_REQUIRES_PROVIDER_STRUCTURE_EVIDENCE'})

for d,files in outer_paths.items():
    for p in files:
        if p.suffix.lower()=='.zip': scan_zip(d,p,p)

inventory=pd.DataFrame(inventory_rows)
assert len(inventory) and inventory.path_safe.all() and not inventory.encrypted.any(),'Unsafe or encrypted archive member'
summary=[]
for d in DATASETS:
    x=inventory[inventory.dataset_id==d]; leaf=x[~x.nested_zip]
    summary.append({'dataset_id':d,'recursive_member_count':len(x),'max_depth':int(x.depth.max()) if len(x) else 0,'image_like_leaf_count':int(leaf.suffix.isin(IMAGE_EXT).sum()),'table_like_leaf_count':int(leaf.suffix.isin(TABLE_EXT).sum()),'nested_zip_count':int(x.nested_zip.sum()),'opaque_nonzip_archive_count':int(x.opaque_rar_or_7z.sum()),'all_paths_safe':bool(x.path_safe.all()) if len(x) else False,'encrypted_member_count':int(x.encrypted.sum()) if len(x) else 0})
archive_summary=pd.DataFrame(summary)
if REPLAY:
    assert canon(pd.read_csv(INVENTORY))==canon(inventory); assert canon(pd.read_csv(ARCHIVE_SUMMARY))==canon(archive_summary)
else:
    write_csv(INVENTORY,inventory); write_csv(ARCHIVE_SUMMARY,archive_summary)
display(archive_summary)


,dataset_id,recursive_member_count,max_depth,image_like_leaf_count,table_like_leaf_count,nested_zip_count,opaque_nonzip_archive_count,all_paths_safe,encrypted_member_count
0,BUS_BRA_2024,3754,0,3750,4,0,0,True,0
1,BUSI_WHU_2025_V3,1867,1,1854,9,2,0,True,0
2,BREAST_LESIONS_USG_2024,522,0,522,0,0,0,True,0
3,BUS_UCLM_2025_V3,1366,0,1366,0,0,0,True,0
4,RODRIGUES_BUI_2017,251,1,250,0,1,0,True,0


In [3]:
# @title 11C-R2-2. Parse released tables and run dataset-specific label/group adapters
def get_bytes(d,virtual):
    e=entry_lookup[(d,virtual)]; return read_chain(e['outer'],e['chain'])
def norm(s): return re.sub(r'[^a-z0-9]+','_',str(s).strip().lower()).strip('_')
def read_table_bytes(name,b):
    suf=Path(name).suffix.lower(); out=[]
    try:
        if suf in {'.csv','.tsv','.txt'}:
            for enc in ['utf-8-sig','utf-8','latin1']:
                try:
                    sep='\t' if suf=='.tsv' else None
                    t=pd.read_csv(io.BytesIO(b),sep=sep,engine='python',encoding=enc); out=[('table',t)]; break
                except Exception: pass
        elif suf in {'.xlsx','.xls'}:
            book=pd.ExcelFile(io.BytesIO(b))
            out=[(sh,pd.read_excel(io.BytesIO(b),sheet_name=sh)) for sh in book.sheet_names]
        elif suf=='.json':
            obj=json.loads(b.decode('utf-8-sig')); out=[('json',pd.json_normalize(obj if isinstance(obj,list) else [obj]))]
    except Exception:
        return []
    return out

tables=[]; parsed_tables={d:[] for d in DATASETS}
for (d,v),e in entry_lookup.items():
    if e['suffix'] in TABLE_EXT and e['size']<=25*1024*1024:
        try: b=get_bytes(d,v)
        except Exception: continue
        parsed=read_table_bytes(v,b)
        for sh,t in parsed:
            parsed_tables[d].append((v,sh,t))
            tables.append({'dataset_id':d,'virtual_path':v,'sheet':sh,'rows':len(t),'columns':len(t.columns),'column_names':' | '.join(map(str,t.columns))[:2000]})
table_audit=pd.DataFrame(tables,columns=['dataset_id','virtual_path','sheet','rows','columns','column_names'])
if REPLAY: assert canon(pd.read_csv(TABLE_AUDIT))==canon(table_audit)
else: write_csv(TABLE_AUDIT,table_audit)

def image_entries(d):
    x=inventory[(inventory.dataset_id==d)&(inventory.suffix.isin(IMAGE_EXT))&(~inventory.nested_zip)]
    return x.virtual_path.astype(str).tolist()
def is_mask_path(v):
    s=v.lower().replace('\\','/')
    return bool(re.search(r'(^|[/_ -])(mask|masks|seg|segs|segmentation|annotation|annotations|gt|ground.?truth)([/_ .-]|$)',s))
def canonical_sample(v):
    p=PurePosixPath(v.split('!/')[-1]); stem=norm(p.stem)
    stem=re.sub(r'(^|_)(mask|segmentation|seg|annotation|gt|ground_truth)(_|$)','_',stem)
    stem=re.sub(r'_+','_',stem).strip('_')
    return stem
def path_label(v):
    s=v.lower()
    if re.search(r'(^|[/_ .-])malignant([/_ .-]|$)',s): return 'malignant'
    if re.search(r'(^|[/_ .-])benign([/_ .-]|$)',s): return 'benign'
    if re.search(r'(^|[/_ .-])normal([/_ .-]|$)',s): return 'normal'
    return ''
def candidate_table_evidence(d,expected_groups=None):
    label=[]; groups=[]; details=[]
    for v,sh,t in parsed_tables[d]:
        if t.empty: continue
        for c in t.columns:
            n=norm(c); vals=t[c].dropna(); nun=int(vals.astype(str).nunique()) if len(vals) else 0
            if re.search(r'pathol|diagnos|class|label|birads|bi_rads|malignan|benign',n):
                label.append((v,sh,str(c),nun,sorted(vals.astype(str).str.lower().unique().tolist())[:20]))
            if re.search(r'patient|subject|case|lesion|study|exam|(^|_)id($|_)',n):
                groups.append((v,sh,str(c),nun))
        details.append((v,sh,len(t),list(map(str,t.columns))))
    group_best=''; group_count=0
    if groups:
        ranked=sorted(groups,key=lambda x:(expected_groups is not None and x[3]==expected_groups, x[3]),reverse=True)
        group_best=' / '.join(map(str,ranked[0][:3])); group_count=ranked[0][3]
        if expected_groups is not None:
            exact=[x for x in groups if x[3]==expected_groups]
            if exact: group_best=' / '.join(map(str,exact[0][:3])); group_count=expected_groups
    return label,groups,group_best,group_count,details

evidence=[]

# BUS-BRA: original/mask pairing plus released CSV label and patient columns.
d='BUS_BRA_2024'; imgs=image_entries(d); originals=[v for v in imgs if not is_mask_path(v)]; masks=[v for v in imgs if is_mask_path(v)]
if not originals and len(imgs)==3750:
    # Fallback for separate image/mask directories with unfamiliar names: pair identical basenames.
    bybase={}
    for v in imgs: bybase.setdefault(canonical_sample(v),[]).append(v)
    originals=[vs[0] for vs in bybase.values()]; masks=[v for vs in bybase.values() for v in vs[1:]]
labels,groups,gb,gc,details=candidate_table_evidence(d,1064)
label_ok=bool(labels); group_ok=(gc==1064)
evidence.append({'dataset_id':d,'pixel_payload_proven':len(originals)>=1000,'original_image_count':len(originals),'mask_or_annotation_count':len(masks),'per_sample_label_mapping_proven':label_ok,'label_basis':('released table: '+str(labels[0][:4])) if labels else 'no diagnostic column parsed','grouping_proven':group_ok,'unique_group_count':gc,'grouping_basis':gb or 'no patient/case column with 1064 unique values','provider_count_reconciled':len(originals)==1875 and gc==1064,'opaque_payload_blocks_qualification':False})

# BUSI-WHU: recurse into BUSI-WHU.zip and parse the released Patient_infos workbook.
d='BUSI_WHU_2025_V3'; imgs=image_entries(d); originals=[v for v in imgs if not is_mask_path(v)]; masks=[v for v in imgs if is_mask_path(v)]
classed=[(v,path_label(v)) for v in originals if path_label(v)]; path_groups={canonical_sample(v) for v,_ in classed}; classes=sorted({y for _,y in classed})
pair_ids={canonical_sample(v) for v in masks}; paired=sum(canonical_sample(v) in pair_ids for v,_ in classed)
table_candidates=[]
for v,sh,t in parsed_tables[d]:
    if t.empty: continue
    label_cols=[]; patient_cols=[]; key_cols=[]
    for c in t.columns:
        n=norm(c); vals=t[c].dropna(); low=set(vals.astype(str).str.strip().str.lower().tolist())
        if re.search(r'pathol|diagnos|class|label|birads|bi_rads|malignan|benign',n) or ({'benign','malignant'}<=low): label_cols.append(str(c))
        if re.search(r'patient|subject|case|lesion|study|exam',n): patient_cols.append(str(c))
        if re.search(r'image|file|name|patient|subject|case|lesion|(^|_)id($|_)',n): key_cols.append(str(c))
    if label_cols and key_cols and len(t)==len(originals):
        ranked=sorted(key_cols,key=lambda c:(bool(re.search(r'patient|subject|case|lesion',norm(c))),int(t[c].dropna().astype(str).nunique())),reverse=True)
        key=ranked[0]; ngrp=int(t[key].dropna().astype(str).nunique())
        table_candidates.append({'virtual_path':v,'sheet':sh,'rows':len(t),'label_columns':label_cols,'key_column':key,'unique_groups':ngrp,'patient_key':bool(re.search(r'patient|subject|case|lesion',norm(key)))})
table_map=sorted(table_candidates,key=lambda x:(x['patient_key'],x['unique_groups']),reverse=True)[0] if table_candidates else None
path_map_ok=set(classes)>={'benign','malignant'} and len(classed)==len(originals)
table_map_ok=table_map is not None and table_map['unique_groups']>=2
mapping_ok=path_map_ok or table_map_ok
group_count=table_map['unique_groups'] if table_map else len({canonical_sample(v) for v in originals})
group_basis=('released Patient_infos workbook '+table_map['virtual_path']+' / '+table_map['key_column']) if table_map else ('released image/annotation canonical basename as lesion key; paired annotations='+str(paired))
label_basis=('released Patient_infos workbook columns: '+' | '.join(table_map['label_columns'])) if table_map else ('benign/malignant tokens in recursively exposed payload paths: '+str(classes))
evidence.append({'dataset_id':d,'pixel_payload_proven':len(originals)>0,'original_image_count':len(originals),'mask_or_annotation_count':len(masks),'per_sample_label_mapping_proven':mapping_ok,'label_basis':label_basis,'grouping_proven':mapping_ok and group_count>=2,'unique_group_count':group_count,'grouping_basis':group_basis,'provider_count_reconciled':len(originals)==927 and mapping_ok,'opaque_payload_blocks_qualification':bool(archive_summary.set_index('dataset_id').loc[d,'opaque_nonzip_archive_count'])})

# BrEaST-Lesions-USG: retain the workbook-based patient/diagnosis evidence and re-check exact receipt.
d='BREAST_LESIONS_USG_2024'; imgs=image_entries(d); folder=RECEIPT_ROOT/FOLDERS[d]
xlsx=next(iter(sorted(folder.glob('*.xlsx'))),None); label_cols=[]; group_cols=[]; group_values=set()
if xlsx:
    book=pd.ExcelFile(xlsx)
    for sh in book.sheet_names:
        t=pd.read_excel(xlsx,sheet_name=sh)
        for c in t.columns:
            n=norm(c); vals=t[c].dropna().astype(str)
            if re.search(r'pathol|diagnos|birads|bi_rads|malignan|benign',n): label_cols.append(str(c))
            if re.search(r'patient|subject|case|anon|(^|_)id($|_)',n): group_cols.append(str(c)); group_values.update(vals.tolist())
evidence.append({'dataset_id':d,'pixel_payload_proven':len(imgs)>0,'original_image_count':len(imgs),'mask_or_annotation_count':0,'per_sample_label_mapping_proven':bool(label_cols),'label_basis':'clinical workbook columns: '+' | '.join(label_cols),'grouping_proven':len(group_values)==256,'unique_group_count':len(group_values),'grouping_basis':'clinical workbook identifier columns: '+' | '.join(group_cols),'provider_count_reconciled':len(group_values)==256,'opaque_payload_blocks_qualification':False})

# BUS-UCLM: author adapter = first four filename characters; labels are decoded from every RGB mask.
d='BUS_UCLM_2025_V3'; imgs=image_entries(d); originals=[v for v in imgs if '/images/' in ('/'+v.lower().replace('\\','/'))]; masks=[v for v in imgs if '/masks/' in ('/'+v.lower().replace('\\','/'))]
if not originals: originals=[v for v in imgs if not is_mask_path(v)]
if not masks: masks=[v for v in imgs if is_mask_path(v)]
orig_by_name={PurePosixPath(v.split('!/')[-1]).name:v for v in originals}; mask_by_name={PurePosixPath(v.split('!/')[-1]).name:v for v in masks}
paired=sorted(set(orig_by_name)&set(mask_by_name)); class_counts={'benign':0,'malignant':0,'normal':0,'ambiguous':0}; decode_errors=[]
for name in paired:
    try:
        arr=np.asarray(Image.open(io.BytesIO(get_bytes(d,mask_by_name[name]))).convert('RGB'))
        red=bool(np.any((arr[...,0]>200)&(arr[...,1]<80)&(arr[...,2]<80)))
        green=bool(np.any((arr[...,1]>200)&(arr[...,0]<80)&(arr[...,2]<80)))
        if red and green: cls='ambiguous'
        elif red: cls='malignant'
        elif green: cls='benign'
        else: cls='normal'
        class_counts[cls]+=1
    except Exception as e: decode_errors.append(name+':'+type(e).__name__)
patient_ids={name[:4] for name in paired if len(name)>=4}
expected_counts=provider_evidence['datasets'][d]['expected_class_counts']
counts_match=all(class_counts[k]==v for k,v in expected_counts.items()) and class_counts['ambiguous']==0
evidence.append({'dataset_id':d,'pixel_payload_proven':len(paired)>0,'original_image_count':len(originals),'mask_or_annotation_count':len(masks),'per_sample_label_mapping_proven':counts_match and not decode_errors,'label_basis':'all paired RGB masks decoded by provider rule; counts='+json.dumps(class_counts,sort_keys=True),'grouping_proven':len(patient_ids)==38,'unique_group_count':len(patient_ids),'grouping_basis':'first four filename characters per author prepare_partitions.py','provider_count_reconciled':len(paired)==683 and counts_match and len(patient_ids)==38,'opaque_payload_blocks_qualification':False})

# Rodrigues: inspect recursively when possible; provider class totals are not enough without per-image class and case mapping.
d='RODRIGUES_BUI_2017'; imgs=image_entries(d); originals=[v for v in imgs if not is_mask_path(v)]; classed=[(v,path_label(v)) for v in originals if path_label(v)]; counts=pd.Series([y for _,y in classed]).value_counts().to_dict() if classed else {}
groupset={canonical_sample(v) for v,_ in classed}; expected_counts=provider_evidence['datasets'][d]['expected_class_counts']; counts_match=all(int(counts.get(k,0))==v for k,v in expected_counts.items())
opaque=bool(archive_summary.set_index('dataset_id').loc[d,'opaque_nonzip_archive_count'])
evidence.append({'dataset_id':d,'pixel_payload_proven':len(originals)>0,'original_image_count':len(originals),'mask_or_annotation_count':0,'per_sample_label_mapping_proven':counts_match and len(classed)==len(originals),'label_basis':'class-bearing recursive paths; counts='+json.dumps(counts,sort_keys=True),'grouping_proven':counts_match and len(groupset)==250,'unique_group_count':len(groupset),'grouping_basis':'released per-image basename used as lesion key only if 250 classed images are exposed','provider_count_reconciled':counts_match and len(originals)==250,'opaque_payload_blocks_qualification':opaque})

evidence_matrix=pd.DataFrame(evidence)
if REPLAY: assert canon(pd.read_csv(EVIDENCE_MATRIX))==canon(evidence_matrix)
else: write_csv(EVIDENCE_MATRIX,evidence_matrix)
display(evidence_matrix)


,dataset_id,pixel_payload_proven,original_image_count,mask_or_annotation_count,per_sample_label_mapping_proven,label_basis,grouping_proven,unique_group_count,grouping_basis,provider_count_reconciled,opaque_payload_blocks_qualification
0,BUS_BRA_2024,True,1875,1875,True,"released table: ('BUSBRA/10-fold-cv.csv', 'tab...",True,1064,BUSBRA/bus_data.csv / table / Case,True,False
1,BUSI_WHU_2025_V3,True,927,927,True,released Patient_infos workbook columns: Benig...,True,816,released Patient_infos workbook BUSI_WHU Breas...,True,False
2,BREAST_LESIONS_USG_2024,True,522,0,True,clinical workbook columns: BIRADS | Diagnosis,True,256,clinical workbook identifier columns: CaseID,True,False
3,BUS_UCLM_2025_V3,True,683,683,True,all paired RGB masks decoded by provider rule;...,True,38,first four filename characters per author prep...,True,False
4,RODRIGUES_BUI_2017,True,250,0,True,"class-bearing recursive paths; counts={""benign...",True,250,released per-image basename used as lesion key...,True,False


In [4]:
# @title 11C-R2-3. Apply corrected prefit gates and freeze the recovered roster
rows=[]; ev=evidence_matrix.set_index('dataset_id'); r1d=r1_decisions.set_index('dataset_id')
for i,d in enumerate(DATASETS,1):
    a=ev.loc[d]
    receipt_ok=bool(reverify.loc[reverify.dataset_id==d,'sha256_matches_r1'].all())
    if not receipt_ok: decision,status,reason='HOLD','HOLD_RECEIPT_CHANGED','Receipt bytes no longer match sealed Stage11C-R ledger'
    elif bool(a.opaque_payload_blocks_qualification) and not bool(a.pixel_payload_proven): decision,status,reason='HOLD','HOLD_OPAQUE_NESTED_FORMAT','Nested non-ZIP payload could not be audited'
    elif not bool(a.pixel_payload_proven): decision,status,reason='HOLD','HOLD_PIXEL_ASSET_STRUCTURE','No auditable image payload after recursive inspection'
    elif not bool(a.per_sample_label_mapping_proven): decision,status,reason='HOLD','HOLD_PER_SAMPLE_LABEL_MAPPING','Provider describes classes but released payload did not yield an auditable label for every sample'
    elif not bool(a.grouping_proven): decision,status,reason='HOLD','HOLD_GROUPING_EVIDENCE','No deterministic patient/lesion grouping key was established'
    elif not bool(a.provider_count_reconciled): decision,status,reason='HOLD','HOLD_PROVIDER_COUNT_RECONCILIATION','Internal payload counts did not reconcile with provider evidence'
    else: decision,status,reason='QUALIFY','QUALIFY_PREFIT_RECURSIVE_PROVIDER_EVIDENCE','Exact receipt, recursive pixels, per-sample labels, deterministic groups, and provider counts reconciled; pixel dedup remains mandatory'
    rows.append({'original_priority':i,'dataset_id':d,'stage11c_r_v01_decision':r1d.loc[d,'decision'],'stage11c_r_v01_status':r1d.loc[d,'status'],'decision':decision,'status':status,'reason':reason,'receipt_bytes_match_r1':receipt_ok,'performance_evaluated':False})
decisions=pd.DataFrame(rows)
qualified=decisions[decisions.decision=='QUALIFY'].copy()
roster=qualified[['original_priority','dataset_id']].copy(); roster.insert(0,'roster_position',range(1,len(roster)+1)); roster['role']='RECOVERED_ORIGINAL_DEVELOPMENT_EXTENSION'; roster['source_axis_fitted']=False; roster['stage12_role_permitted']=False
authorised=len(roster)>=MINIMUM
decision='SEAL_STAGE11C_R2_AUTHORISE_STAGE11D_R_DEDUP_GROUPED_SPLIT_FREEZE_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED' if authorised else 'HOLD_STAGE11C_R2_INSUFFICIENT_QUALIFIED_DOMAINS_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
if REPLAY:
    assert canon(pd.read_csv(DECISIONS))==canon(decisions); assert canon(pd.read_csv(ROSTER))==canon(roster); handoff=verify_self(HANDOFF,'handoff_sha256')
else:
    write_csv(DECISIONS,decisions); write_csv(ROSTER,roster)
    handoff={'stage':'Stage11C-R2','target':'Stage11D-R','decision':decision,'protocol_seal_sha256':seal['seal_sha256'],'parent_stage11c_r_final_sha256':r1['final_record_sha256'],'provider_registry_sha256':provider_evidence['registry_sha256'],'receipt_reverification_sha256':sha_file(RECEIPT_REVERIFY),'recursive_inventory_sha256':sha_file(INVENTORY),'evidence_matrix_sha256':sha_file(EVIDENCE_MATRIX),'readjudication_sha256':sha_file(DECISIONS),'qualified_roster_sha256':sha_file(ROSTER),'qualified_dataset_ids':roster.dataset_id.tolist(),'qualified_domain_count':len(roster),'minimum_feasible_domains':MINIMUM,'stage11d_r_authorised':authorised,'next_stage_boundary':'cross-roster pixel deduplication, exact image-label-group manifest freeze, grouped split freeze, then development-only source recoverability','stage12_authorised':False,'ddo2_fitted':False,'locked_blind_assets_touched':False}
    handoff['handoff_sha256']=sha_json(handoff); write_json(HANDOFF,handoff)
display(decisions[['dataset_id','stage11c_r_v01_status','decision','status','reason']]); print('Qualified roster:',roster.dataset_id.tolist()); print('Stage11D-R authorised:',authorised)


,dataset_id,stage11c_r_v01_status,decision,status,reason
0,BUS_BRA_2024,HOLD_LABEL_SEMANTICS,QUALIFY,QUALIFY_PREFIT_RECURSIVE_PROVIDER_EVIDENCE,"Exact receipt, recursive pixels, per-sample la..."
1,BUSI_WHU_2025_V3,HOLD_PIXEL_ASSET_STRUCTURE,QUALIFY,QUALIFY_PREFIT_RECURSIVE_PROVIDER_EVIDENCE,"Exact receipt, recursive pixels, per-sample la..."
2,BREAST_LESIONS_USG_2024,QUALIFY_PREFIT_MANUAL_OFFICIAL_RECEIPT,QUALIFY,QUALIFY_PREFIT_RECURSIVE_PROVIDER_EVIDENCE,"Exact receipt, recursive pixels, per-sample la..."
3,BUS_UCLM_2025_V3,HOLD_LABEL_SEMANTICS,QUALIFY,QUALIFY_PREFIT_RECURSIVE_PROVIDER_EVIDENCE,"Exact receipt, recursive pixels, per-sample la..."
4,RODRIGUES_BUI_2017,HOLD_PIXEL_ASSET_STRUCTURE,QUALIFY,QUALIFY_PREFIT_RECURSIVE_PROVIDER_EVIDENCE,"Exact receipt, recursive pixels, per-sample la..."


Qualified roster: ['BUS_BRA_2024', 'BUSI_WHU_2025_V3', 'BREAST_LESIONS_USG_2024', 'BUS_UCLM_2025_V3', 'RODRIGUES_BUI_2017']
Stage11D-R authorised: True


In [5]:
# @title 11C-R2-4. Run independent replay, leakage, and authority checks
checks=[]
def ck(name,passed,evidence): checks.append({'check':name,'passed':bool(passed),'evidence':str(evidence)[:2000]})
ck('sealed Stage11C-R parent exact',r1['final_record_sha256']==EXPECTED_R1_FINAL,r1['final_record_sha256'])
ck('parent result preserved as hold',r1['stage11d_r_authorised'] is False,r1['decision'])
ck('five originals exact order',decisions.dataset_id.tolist()==DATASETS,decisions.dataset_id.tolist())
ck('receipt bytes exactly match parent ledger',reverify[['size_matches_r1','sha256_matches_r1']].all().all(),reverify.sha256.tolist())
ck('provider evidence registry self sealed',verify_self(PROVIDER_REGISTRY,'registry_sha256')['registry_sha256']==provider_evidence['registry_sha256'],provider_evidence['registry_sha256'])
ck('recursive paths safe and unencrypted',inventory.path_safe.all() and not inventory.encrypted.any(),len(inventory))
ck('no locked-blind token in inventory',not inventory.virtual_path.str.lower().str.contains('|'.join(LOCKED),regex=True).any(),LOCKED)
ck('no reserve accessed',runtime['reserves_accessed'] is False,'false')
ck('no runtime provider request',runtime['provider_requests_during_runtime'] is False,'false')
ck('no performance input',not decisions.performance_evaluated.astype(bool).any() and runtime['performance_evaluated'] is False,'all false')
ck('qualify requires every prefit gate',all(bool(r.pixel_payload_proven and r.per_sample_label_mapping_proven and r.grouping_proven and r.provider_count_reconciled) for r in evidence_matrix[evidence_matrix.dataset_id.isin(qualified.dataset_id)].itertuples()),qualified.dataset_id.tolist())
ck('roster equals qualified originals',roster.dataset_id.tolist()==qualified.dataset_id.tolist(),roster.dataset_id.tolist())
ck('minimum rule exact',bool(handoff['stage11d_r_authorised'])==(len(roster)>=MINIMUM),len(roster))
ck('DDO2 and Stage12 prohibited',runtime['ddo2_fitted'] is False and handoff['stage12_authorised'] is False,'false')
ck('locked blind untouched',runtime['locked_blind_assets_touched'] is False and handoff['locked_blind_assets_touched'] is False,'false')
ck('handoff self hash',sha_json({k:v for k,v in handoff.items() if k!='handoff_sha256'})==handoff['handoff_sha256'],handoff['handoff_sha256'])
validity=pd.DataFrame(checks)
if REPLAY: assert canon(pd.read_csv(FIREWALL))==canon(validity)
else: write_csv(FIREWALL,validity)
failed=validity.loc[~validity.passed,'check'].tolist(); assert not failed,'Validity failure: '+'; '.join(failed)
print(f'Independent checks: {validity.passed.sum()}/{len(validity)} passed')


Independent checks: 16/16 passed


In [6]:
# @title 11C-R2-5. Seal report, integrity manifest, final record, and print handoff
q=int((decisions.decision=='QUALIFY').sum()); h=int((decisions.decision=='HOLD').sum()); e=int((decisions.decision=='EXCLUDE').sum())
next_step='BUILD_STAGE11D_R_CROSS_ROSTER_DEDUP_EXACT_MANIFEST_AND_GROUPED_SPLIT_FREEZE' if authorised else 'REVIEW_ONLY_THE_REMAINING_R2_TYPED_HOLDS_OR_PRESEALED_RESERVE_EVIDENCE_WITHOUT_PERFORMANCE_BASED_SUBSTITUTION'
report=f"""# Stage 11C-R2 report

## Answer first

- Exact Stage 11C-R manual receipts reverified: **{int(reverify.sha256_matches_r1.sum())}/{len(reverify)} files**.
- Qualified / held / excluded domains: **{q} / {h} / {e}**.
- Minimum feasible domains: **{MINIMUM}**.
- Stage 11D-R authorised: **{authorised}**.
- Decision: `{decision}`.

## Corrected dataset adjudication

{markdown_table(decisions)}

## Evidence matrix

{markdown_table(evidence_matrix)}

## Method boundary

The sealed Stage 11C-R v0.1 result was not overwritten. This readjudication used only the same five exact receipt byte streams, recursively exposed payload content, and the pre-performance official provider/author evidence registry. No reserve, embedding, source AUC, transfer result, DDO2 operation, Stage 12 operation, or locked-blind asset was used. Stage 11D-R, if authorised, must still perform cross-roster pixel deduplication and freeze exact patient/lesion-grouped manifests before any fitting.
"""
if REPLAY: assert REPORT.read_text(encoding='utf-8')==report
else: write_text(REPORT,report)
tracked=[PROTOCOL,PROVIDER_REGISTRY,PARENT_COMMIT,RECEIPT_REVERIFY,INVENTORY,ARCHIVE_SUMMARY,TABLE_AUDIT,EVIDENCE_MATRIX,DECISIONS,ROSTER,HANDOFF,FIREWALL,REPORT]
manifest=pd.DataFrame([{'relative_path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha_file(p)} for p in tracked])
if REPLAY:
    old=pd.read_csv(MANIFEST); assert canon(old)==canon(manifest)
    for r in old.itertuples():
        p=ROOT/r.relative_path; assert p.is_file() and p.stat().st_size==int(r.size_bytes) and sha_file(p)==r.sha256
else: write_csv(MANIFEST,manifest)
payload={'stage':'Stage11C-R2','version':'0.1','decision':decision,'protocol_seal_sha256':seal['seal_sha256'],'parent_stage11c_r_final_sha256':r1['final_record_sha256'],'provider_registry_sha256':provider_evidence['registry_sha256'],'receipt_reverification_sha256':sha_file(RECEIPT_REVERIFY),'recursive_inventory_sha256':sha_file(INVENTORY),'evidence_matrix_sha256':sha_file(EVIDENCE_MATRIX),'typed_readjudication_sha256':sha_file(DECISIONS),'qualified_roster_sha256':sha_file(ROSTER),'stage11d_r_handoff_sha256':handoff['handoff_sha256'],'output_integrity_manifest_sha256':sha_file(MANIFEST),'qualified_domains':q,'held_domains':h,'excluded_domains':e,'minimum_feasible_domains':MINIMUM,'stage11d_r_authorised':authorised,'stage12_authorised':False,'ddo2_fitted':False,'locked_blind_assets_touched':False,'next_step':next_step}
if REPLAY:
    final=verify_self(FINAL,'final_record_sha256')
    for k,v in payload.items(): assert final[k]==v,f'Replay final changed: {k}'
else:
    final=dict(payload); final['completed_utc']=now(); final['final_record_sha256']=sha_json(final); write_json(FINAL,final)
runtime.update({'completed':True,'decision':decision,'final_record_sha256':final['final_record_sha256'],'last_updated_utc':now()}); RUNTIME.write_text(json.dumps(runtime,indent=2)+'\n')
print('================ STAGE 11C-R2 COMPLETE ================')
print('Exact manual receipt files reverified:',f"{int(reverify.sha256_matches_r1.sum())}/{len(reverify)}")
print('Recursive image leaves by domain:',dict(zip(archive_summary.dataset_id,archive_summary.image_like_leaf_count)))
print('Qualified / held / excluded:',f'{q}/{h}/{e}')
print('Qualified development roster / minimum feasible:',f'{len(roster)}/{MINIMUM}')
print('Stage11D-R authorised:',authorised)
print('Decision:',decision)
print('Protocol seal:',seal['seal_sha256'])
print('Provider evidence registry hash:',provider_evidence['registry_sha256'])
print('Receipt reverification hash:',final['receipt_reverification_sha256'])
print('Recursive inventory hash:',final['recursive_inventory_sha256'])
print('Evidence matrix hash:',final['evidence_matrix_sha256'])
print('Typed readjudication hash:',final['typed_readjudication_sha256'])
print('Qualified roster hash:',final['qualified_roster_sha256'])
print('Stage11D-R handoff hash:',final['stage11d_r_handoff_sha256'])
print('Final record hash:',final['final_record_sha256'])
print('Next step:',next_step)


================ STAGE 11C-R2 COMPLETE ================
Exact manual receipt files reverified: 6/6
Recursive image leaves by domain: {'BUS_BRA_2024': 3750, 'BUSI_WHU_2025_V3': 1854, 'BREAST_LESIONS_USG_2024': 522, 'BUS_UCLM_2025_V3': 1366, 'RODRIGUES_BUI_2017': 250}
Qualified / held / excluded: 5/0/0
Qualified development roster / minimum feasible: 5/4
Stage11D-R authorised: True
Decision: SEAL_STAGE11C_R2_AUTHORISE_STAGE11D_R_DEDUP_GROUPED_SPLIT_FREEZE_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED
Protocol seal: 12ca6c10e436e75efbb5bbca9e956f14f877bc1b8c646867060c9f8a6c326e25
Provider evidence registry hash: 248b85844bbcb2c8dcad75e6326aa483ce9224047fee934fcd7da2602bacdc3b
Receipt reverification hash: 29ecbfff1e15d4ffd9ad9133f7319c124f6e8142aab5ae797f06ae4ce5438111
Recursive inventory hash: ecf0ed58bfb1baf1a3c44b8b7364fc4129b713d7122016ba9276e2264f86ebe9
Evidence matrix hash: a02a147e9d1fe2e73559c018e2e6c44362220e84ec785348f90ef898e48de854
Typed readjudication hash: 1cf1530213e2e617cd1db2e5ef